
---
tags: ssh, globus, expanse
---

# Getting Started
## People
- Cindy Wong
- Andre Zonca - Chair of summer institute
- Susan Rathbun - Program Manager
- Nicole Wolter - Computational and Data Science Research Specialist
- Marty
- Igor
- Andre
- Linton
- Vatia
- Ethan
- Florida
- 

## Location
9836 Hopkins Drive, La Jolla CA, 92093

Code in Park Mobile: 
- 47800 
- $8 instead of $32
- Tickets are $80

## Details and Resources
- Allocation: CIS261077
- Reservations: si26cpu OR si26gpu
- Account:sdp173
- Materials: https://github.com/sdsc/sdsc-summer-institute-2026/tree/main
- Slack: https://app.slack.com/client/T0BHCRCNU68/C0BH2SBMPH9
- Videos:  https://www.sdsc.edu/education/on-demand-           
  learning/index.html
- git repos
  - https://github.com/sdsc/sdsc-summer-institute-2026.git
  - https://github.com/sdsc-hpc-training-org/basic_skills

Profile Link: https://allocations.access-ci.org/login
- Use ACCESS CI login not Cal Poly's CI logon (they are different)
- User: **jkrone**
- Password: see lastpass
- identity name: jkrone@access-ci.org
- subject id: f82c18ec-dd44-44e0-af07-f091684fceb6


### Login Node Access
Login nodes are meant for file editing, simple data analysis, and other tasks that use minimal compute resources. Use batch nodes for computationally intensive work.

```shell
# create ssh key
ssh-keygen -t ed25519 -C "jmkrone@calpoly.edu"

# add or change passphrase
ssh-keygen -p -f ~/.ssh/id_ed25519

# start ssh agent
eval "$(ssh-agent -s)"

# load key into agent
ssh-add ~/.ssh/id_ed25519_expanse

# verify key was added to agent
ssh-add -l

# setup MFA
# Login using Globus and Access ID
https://passive.sdsc.edu
# Add SSH key and MFA app in the website
https://passive.sdsc.edu/client/

# add your git key to the ssh-agent
ssh-add ~/.ssh/id_ed25519_github

# pass identities associated with ssh-agent through to remote
# do not copy ssh-key onto shared remote system
ssh -A jkrone@login.expanse.sdsc.edu

# setup host file b/c correct key may not be chosen by default
vim ~/.ssh/config

### On Local Machine
Host expanse
    HostName login.expanse.sdsc.edu
    User jkrone
    IdentityFile ~/.ssh/id_ed25519_expanse
  
Host github.com
   HostName github.com
   User git
   IdentityFile ~/.ssh/id_ed25519_github

### On Remote Machine (note the deliberate lack of identity file)
Host github.com
    HostName github.com
    User git

```
### Expanse User Portal
The [Expanse User Portal](https://portal.expanse.sdsc.edu/pun/sys/dashboard) provides a quick and easy way for Expanse users to log in, transfer and edit files, and submit and monitor jobs. The Portal provides a gateway for launching interactive applications such as MATLAB, RStudio, and an integrated web-based environment for file management and job submission. All ACCESS users with a valid Expanse allocation have access via their ACCESS-based credentials.


## Basic Linux Skills for Expanse
Fundamental Linux commands for navigating the file system, managing files, and understanding permissions.
```shell
date
hostname
whoami
env
groups

# recursively copy and preserve data and time
cp -r -p

# list by date
ls -alt

head -n 1
tail -n 1

# get line number and ignore case
grep -ni

# change permissions
chmod 660 *

# change group
chgrp heart *.out
```

Other utilities to learn
- grep, sort, tar, gzip ,and pigz

## Basic Orientation on Expanse

```shell
expanse-client user
expanse-client project sdp173
expanse-client project sdp173 -v
```
### Notes
You should use a **reservation** to get high priority in the queue
- si26cpu for most jobs
- si26gpu for GPU jobs

## Interactive Computing on Expanse
Requesting and using interactive sessions on CPU and GPU nodes.

### Request Interactive CPU Node
This node could be used for running jobs, jupyter notebooks, etc.

The following example will request one regular compute node, 4 cores, in the debug partition for 30 minutes.
```shell
srun --partition=debug  --pty --account=<<project>> --nodes=1 --ntasks-per-node=4 --mem=8G -t 00:30:00 --wait=0 --export=ALL /bin/bash
```

### Request Interactive GPU Node


```shell
srun --partition=gpu-shared --reservation=gputraining --nodes=1 --ntasks-per-node=6 --gres=gpu:k80:1  -t 03:00:00 --pty --wait=0 /bin/bash

# Once in the allocation, Load the CUDA and PGI compiler modules

module purge
module load gnutools
module load cuda
module load pgi

# If you get a license error when loading gpi compiler
export LM_LICENSE_FILE=40200@elprado.sdsc.edu:$LM_LICENSE_FILE
```

## Day 1

### Parallel Computing
- Very important
- Use atomic ints for counters
- Use mutexes for critial sections
- Data partioning is important for paralelizing
- Shared memory is helpful (internal memory aka registers)
- Cache 1, 2 Side 46
- You MUST be in level 2 cache, otherwise you're underutilizing the system
#### Summary

### Expanse
- https://expanse.sdsc.edu
- For jobs that run on one rack
- Liquid cooled rack
- Storage in Petabytes
- Hardware does fail. Have a hello world batch job to test
#### HPC System Architecture
- Login Node - simple tasks
- Computes Note - allocated by scheduler
- Internal Network - high bandwidth
- Shared network Filesystems - I/O
- Interactive Computing vs Batch Processing

#### Jupyter on Expanse
Install Galyleo

```shell
# Set env var to application that is aready installed
export PATH="/cm/shared/apps/sdsc/galyleo:${PATH}"

# helps you see which modules are avaliable - some have more libraries installed
module avali

# works
galyleo launch --account sdp173 --reservation si26cpu --partition compute --cpus 2 --memory 4 --time-limit 00:30:00 --env-modules cpu/0.17.3b,gcc/10.2.0,py-jupyterlab/3.2.1

# does not work
galyleo launch --account sdp173 --reservation si26cpu --partition compute --cpus 2 --memory 4 --time-limit 00:30:00 --env-modules cpu/0.17.3b,gcc/10.2.0,anaconda3/2021.05/q4munrg


```

### 2.3 High Throughput Computing
- Complecs
- HPC = Supercomputers and computer clusters to solve advanced computation problems
- Speed - 
- Scale - larger problems
- Throughput - solve many simple problems quickly

Usually - 2 CPUs per nodes on Expanse so parallelism is crucial!

#### High-Throughput Computing (HTC)
- Embarassingly parallel problems (small independent subtasks)
- Could run on laptops, but need to do it millions of times
- Brut-force, Batch processing, Montecarlo simulations
- Parameter sweeps

Advantags of HTC vs. HPC
- Simpler programming models - shared memory, serial
- Leverage distributed and heterogeneoud resources easily
- Resilient against job failures

#### Many-Task Computing (MTC)
- Distinct subtasks of variable complexity, often coupled with I/O operations (File Modeling Workflows)

#### Batch Scheduling
- #SBATCH directives tell scheduler these are the resources I want

#### Job Arrays
- Spinup multiple versions of the same job in one script `#SBATCH: array=1`

#### Job Dependencies
- Link post or pre processing steps to job

#### Job Bundling
- Nothing in slurm that will do it natively
- Most HPC systems don't allow sharing nodes (Expanse does allow partial node jobs)
- **Resource scheduling at the node-level**
- Allows more effective utilization

#### Distributed HTC Resources
- Open Science Grid (OSG) (Use to be PATh)
    - OS Pool is Free
    - Under 12 hour jobs
    - compute resources (CERN), HTCondor = scheduler
    - All jobs are small
    - portal.osg-htc.org
- National Research Platform
    - Nautilus
    - Lots of GPUs

#### Tools

Dask (parallel pandas)
- parallel and distributed computing. Custom schedulers that execute task graphs

Pegasus WMS
- Describe workflow in python and execute (like CDK)

Snakemake
- Bioinformatics
- Interact with slurm

Nextflow
- Bioinformatics
- Interact with slurm

Gnu Parallel
- https://hpc.njit.edu/Software/utilities/parallel/

#### Take Away

A lot of python libs are parallelized already you just need to turn on those options

#### Hands On

- Job Arrays

### 2.4 Modules

```shell
# Path to module on system
$MODULEPATH

# Search specific module in system
module spider gcc

# Load modules
module load <application name>      # can put in .bashrc
module unload <application name>

# See loaded modules
module list

# Reset default modules
module reset

# Unload all modules
module purge

# Show module details
module show <application name>
```

#### Singularity
- Container is a static file that includes executable code
- Expanse has singularity containers
- Must build singularity containers on your laptop b/c it requires root access (can't get on expanse)

#### Python
- Use conda and mamba to create virtual environments to run program in

#### Install from source
cmatrix

Environment variables
• PATH: where to look for executables
• LD_LIBRARY_PATH: where to look for shared libraries
• CPATH: where to look for header and include files
➢ Other environment variables sometimes needed by various software
• LIBRARY_PATH, C_LIBRARY_PATH, LD_INCLUDE_PATH
• LDFLAGS, LDLIBS
➢ Environment Module
• Application that load/unload other applications on demand.
• Most ACCESS supercomputing sites use modules. Much more convenient than setting
variables in ~/.bashrc
➢ Two types of Environment Modules
• Lmod: a Lua-based environment module system (Expanse, TSCC and other SDSC HPC)
• Tcl: an environment module in Tool Command Language (less common)



# Day 2
## 3.1 Data Management
[Agenda](https://github.com/sdsc/sdsc-summer-institute-2026/blob/main/3.1_data_management/README.md)
### Data Storage and File Management
- Your HOME directory is the only directory that is backedup on HPCs
- The Filesystem Hierarchy is not the file system (it's a norm)
- The Filesystem is an operating system between the OS and files
- Using `df -Th` an looking at the home directories helps understand potential impact of thrashing
- No universal way to check your quota on a filesystem
- Estimate avaliable file space currently being used `du`
- How much **memory** is avaliable with `free -h`
- Some frameworks generate TONS of files (OpenFOAM is notoriously bad for this). AI/Model tracking frameworks can also be bad.

#### Memory Hierarchy
- registers - 1 cycle , on CPU, volitile
- caches - 10 cycles , on CPU, volitile
- main memory - 100 cycles (here and up is good. Below is bad for HPC), off CPU, volitile
- flash disk - 1 Million cycles, non-volitile
- traditional disk - 10 Million cycles, non-volitile
- remote secondary storage (e.g. memory), non-volitile

#### Filesystem Structures
- Blocks - 
- Inodes - data structure that describes a filesystem object such as a file or directory
  - There is a limit to the number of inodes
- File - set of blocks use to store data

#### Lustre filesystem
- **parallel distributed filesystem**
- Used for large file I/O - reading/writing very large files
- client -> meta data server -> storage
- `lfs` - cli utility for Lustre filesystems
- Scratch file system on expanse is Lustre
- Can also use to check your quota(s), number of files
- `lfs quota -u $USER -h /expanse/lustre/scratch/$USER/templ_project`

#### NSF filesystem
- Used for **Home directories**
- popular **distributed filesystem** protocol in use on HPC systems

#### ext4
- default **local filesystem** for most linux OS distros

#### Ceph
- **distributed storate** that provides block, file, and object-based storage
- More resilient to hardware failures than Lustre

#### Vast
- next gen Luster
- Can deal with small files

### Take Away
- **Move your data onto the right filesystem**
- Transforming ImageNet to TFRecords (150-200GB w/ tons of files)
- If the filesystem is distributed, you're making a network call
- unzipped and ran on multiple filesystems
- /scratch - ext4 = 1 hour
- /home - nfs = 10 hours
- /expanse/luster  = 100 hours
- /expanse/ceph = 10 hours
- /expanse/vast = 1.5 hours
- /cvmfs (osdf) = 10 hours

## Data Transfer and Tools
- Souce
- Format
- Size
- Destination
- Method
- **Data Integrity** is verified with **Checksums** (md5 and sha256sum)
- **Data compression**
  - Lossless compression - can reconstruct file completely
  - Lossy compression - lose data when shrinking b/c data precision is not needed (picture compression)
  - `gzip` for compression/decompression
  - `pigz` parallel gzip (and gzip uses pigz under the hood)
  - `bzip2`
  - `tar` file archiving (organize & compress) tool that uses the other compression tools under the hood
- **Data Transfer Tools**
  - `scp` - secure copy for moving files between remote systems
  - `sftp` - interactive version of `scp`
  - `rsync` - transferring and synchronizing file between two computer systems over a network
    - git can only store/transfer about 1Gb. rsync is generic and works off timestamps
    - can configure to use checksums or timestamps
  - **globus (GridFTP)**
    - non-profit run by University of Chicago
    - cyberinfrastructure products and services for research
    - fast, secure, reliable way to transfer large amounts of data - up to petabytes
    - **Need to setup globus endpoint - not over the public internet!!!!!!!!!!!!!!!!!!!!**
    - Has a web app and commandline tools
  - **rclone**
    - swiss army knife for object storage
    - can replace rsync with `rclone sync source:sourcefile dest:destpath`

### wget
```shell
# downloads file
wget https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz

# downloads all links in a text file
wget -i cifar-100.txt

# resume download
wget -c https://www.image-net.org/data/ILSVRC/2012/ILSVRC2012_img_train.tar
```

### Curl
- more complex than wget but does the same things and more
```shell
# downloads file
curl -O https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz

# downloads all links in a text file
 curl -K cifar-100.txt

# resume download
curl -C - -O https://www.image-net.org/data/ILSVRC/2012/ILSVRC2012_img_train.tar

# download in parallel
curl -Z -O 'https://www.cs.toronto.edu/~kriz/cifar-100-{python,matlab,binary}.tar.gz'
```

### aria2c
- more complex than wget but does the same things and more
```shell
# downloads file
aria2c https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz

# downloads all links in a text file
aria2c -i cifar-100.txt

# resume download
aria2c -c https://www.image-net.org/data/ILSVRC/2012/ILSVRC2012_img_train.tar

# Download a file in parallel with multiple connections per host
$ aria2c -x2 https://www.image-net.org/data/ILSVRC/2012/ILSVRC2012_img_train.tar
```

## Data Distribution Networks and Federations
Dedicated storage resources
- Open Science Data Federation (OSDF)
- Open Storage Network (OSN)
- National Science Data Fabric (NSDF)

### Hands-On
#### Downloading data from the internet
Cool picture
[Activity](https://github.com/sdsc/sdsc-summer-institute-2026/blob/main/3.1_data_management/tutorials/download.md)

1. download CIFAR-10 dataset using `wget`
2. use `tar` to unzip
3. use `md5` to verify data integity
4. use `sha265sum` to verify data integrity

#### Filesystems
[Activity](https://github.com/sdsc/sdsc-summer-institute-2026/blob/main/3.1_data_management/tutorials/filesystem.md)
1. Clone the Dataset to Your Working Directory
   - Trick: **don't download to your home directory** using git. It will take **37 minutes**.
2. Start an interactive session on a compute node
   - `alias start-interactive='srun --partition=shared --account=sdp173 --reservation=si26cpu --nodes=1 --ntasks-per-node=1 --cpus-per-task=2 --mem=4G --time=00:30:00 --pty --wait=0 /bin/bash'`
3. Get your Slurm Job Id and change to your scratch directory
   - cd "/scratch/${USER}/job_${SLURM_JOB_ID}"
4. Clone the repo to your scratch space
   - It will take **2 seconds!!!!!!!!!!!!**
5. Compare Home vs Scratch filesystem types
   - df -Th /scratch
     - /dev/nvme0n1p1 ext4  916G  607M  869G   1% /scratch
   - df -Th /home
     - /etc/auto.home autofs     0     0     0    - /home
     - Doesn't show anything? Why?
   - df -Th "/home/${USER}"
     - 10.22.100.114:/pool4/home/jkrone nfs   212T   36T  177T  17% /home/jkrone
   - Your /home directory is automounted! It's being served from a Network File System (NFS).
   - **automounter** = Just in Time memory allocation
   - **Take Away** - Downloading the data to the /scratch directory used the local /scratch disk on the compute node, while downloading the data to our working directory in /home utilizeed the NFS filesystem, which is a distributed network filesystem.

6. Inspecting the Data
   - Our downloaded repo is unzipped, so we need to zip and copy it back to our HOME directory. Scratch is transient
   - Multiplying by 10 category-level directories, we see that the dataset has approximately 60K raw `.jpg image files. Is that a lot? Hint: Think about how much metadata may be associated with a large number of files.

#### Globus (getting started)
[Activity](https://docs.globus.org/guides/tutorials/manage-files/transfer-files/)
- Easy way to access collections and transfer files
- ESNet demo data transfer nodes - https://fasterdata.es.net/performance-testing/DTNs


## HPC UC San Diego Ecosystem
Expanse
- 10 Million 2019 + 5 million in 2024
- production system
- general-purpose computing plus accelerators

Voyager
- 5 million in 2020
- test bed - alpha & beta users invited to use machine
- AI architecture - purpose build training and infrenece

Cosmos
5 million in 2024
- test bed - alpha & beta users invited to use machine
- Unified memory - processors and accelerators, share memory
- Great because CPU and GPU access same memory so no copying from storage to CPU to GPU

**Access** - National computing ecosystem
- Apply  once and request allocations on the appropriate system

HPC vs Cloud Resource
- Interconnect
- Compute nodes exchange messages through the interconnection.

How to read a rack
- Each slot is a computing node described by Cabinet number and node number

Aircooled
- Cooling at the front, exhaust from the back, Air Handler (curtains seprate hot and colder areas)
- Heat moves to chilled water
- Good for powerloss because more time to deal with issue

Liquid cooled
- cold plates, coolant distribution unit

Notes
- Slingshot - Interconnect
- Infiniband - MPI

mold
biological grouth

Cheaper to decomm the system vs try to maintain b/c it's like 5 million to maintain (power/maint/human cost)

## 3.2 User Support and Onboarding
Best Practices and **Common Issues**

HPC Ecosystems
- login nodes
- File systems
- Compute nodes
- System Admin, Security, Networking, support teams

Office Hours
Research computing consultants

Containers 
- created to support rare software

Environment Modules 
- created for common software
- `module list` - list currently installed software
- `module-spider`- list avaliable software and versions on the system

---------------------------------**Homegrown Tool for Account Info**-----------------------

expanse-client

Allows querying the user statistics.

Usage:
  expanse-client [command]

Available Commands:
  completion  Generate the autocompletion script for the specified shell
  help        Help about any command
  project     Get project information
  resource    Get resources
  user        Get user information

Flags:
  -a, --auth      authenticate the request
  -h, --help      help for expanse-client
  -p, --plain     plain no graphics output
  -v, --verbose   verbose output

Use "expanse-client [command] --help" for more information about a command.

----------------------------------------------------------------------------------------------

## 3.3 Parallel Computing

This session provides an introduction to parallelism by means of OpenMP and MPI. The basic concepts are presented in a lecture format, with hands-on exercises intermixed at the opportune points. During the hands-on sessions, students will be expected to apply what they learned while listening to the lecture. Most exercises are C++ based, with some optional Fortran exercises available, too.

https://github.com/sdsc/sdsc-summer-institute-2026/blob/main/3.3_parallel_computing_mpi_openmp/sdsc_SI26_OpenMP_MPI.pdf

### Enter MPI and OpenMP
You likely want both, as they address complementary needs

**OpenMP – In-process parallelization (aka multi-threading)**
- Allows you to use multiple compute cores
- While having a common view of the data
- Great when multiple threads reuse the same data
- But be careful when writing shared data (i.e. conflicts/synchronization)

**MPI – Multi-process coordination**
- Allows you to use multiple nodes
- No shared memory, so explicit memory partitioning/replication needed
- Explicit information exchange, using library-handled messages

#### OpenMP

- Open Multi-Processing (but actually multi-threading inside a single process)
- OpenMP is a specification (current version is 6.0, but 5.0 is widely supported)
- www.openmp.org

**OpenMP Loop**

```shell
# completes in 48 steps
for(int i=0; i<48; i++){
    z[i] = a*x[i]+y[i];
}

# completes in 4 steps!!!!!!!!!!
#pragma omp parallel for
for(int i=0; i<48; i++){
    z[i] = a*x[i]+y[i];
}

# completes in 4 steps but work is explicitly partitioned statically (before program runs)
#pragma omp parallel for schedule(static,4)
for(int i=0; i<48; i++){
    z[i] = a*x[i]+y[i];
}

```

- Make a **data-parallel problem** use parallel computing
- Good compilers will already vectorize this code on a single core
- **pragma**
  - Adding a `#pragma omp parallel for` will parallelize the folling code/loop

**OpenMP Unbalanced Loop**

```shell
# completes in 4 steps but work is explicitly partitioned statically (before program runs)
#pragma omp parallel for schedule(static,4)
for (int i = 0; i < N; i++) {
    z[i] = a * x[i] + y[i];
        if ((i%12)<8) {
        w[(i/12)*8+ i%12] = b * x[i] + y[i];
    }
}

# completes in 4 steps but work is explicitly partitioned dynamically (at runtime, next element picked at runtime)
#pragma omp parallel for schedule(dynamic,4)
for (int i = 0; i < N; i++) {
    z[i] = a * x[i] + y[i];
        if ((i%12)<8) {
        w[(i/12)*8+ i%12] = b * x[i] + y[i];
    }
}
```

**Reductions - Mix and Match**
- spawn and synchronize, repeat

```shell
wsum=0.0;
foundeq = false;
#pragma omp parallel for reduction(+:wsum) reduction(|:foundeq)
for (int i = 0; i < n; i++) {
    z[i] = a * x[i] + y[i];
    wsum += x[i]*y[i];
    foundeq |= ((b * x[i] + y[i]) == (x[i] + b*y[i]));
}
if (foundeq) printf(“%f\n”, wsum);
```

**OpenMP Atomic Updates**

- can use CPP atomic ints or use `#pragma omp atomic update`
- This creates thread-safe sections

```shell
#pragma omp parallel for
for (int i = 0; i < n; i++) {
    z[i] = a * x[i] + y[i];
    if ((x[i] + b * y[i]) == 0.0) {
        #pragma omp atomic update
        e7[i % 7] += z[i];
    }
}
```

**Handling Large Scratch Spaces**

Note **sgemm**
See slides

**OpenMP nested loops**

- Specify the number of nested loops to parallelize

```shell
// z is 3D, x and y are 2D
#pragma omp parallel for collapse(3)
for (int k = 0; k < P ; k++) {
    for (int j = 0; j < M; j++) {
        for (int i = 0; i < N; i++) {
            z[(k*M+j)*N+i] = a * x[j*N+i] + b *y[k*M+j];
        }
    }
}
```

**Compiling and running OpenMP programs**
Most compilers don’t enable OpenMP by default
- You must explicitly pass a “enable OpenMP flag”
- Most popular compilers use
  - `fopenmp`
- A OpenMP compiled executable will use all available CPU cores
- Auto-detected
- If you want to cap the number of cores in use, set
  - `export OMP_NUM_THREADS=<MAX CORES>`
- When using only a subset of cores, Linux will pick a subset for you
- You can explicitly pick the ones you like with
  - `taskset -c <core range> <your executable and arguments>`

#### Hands-On

1_axpy
- How does optimization flag impact
  - optimizing, time got better but iterations did not
- How does the runtime change, if you change buffer size vs iterations
- 2 sec base vs 0 optimization flags(0.0889 seconds) vs (parallel)0.0126 seconds
2_pi
-  1.6489 seconds
-  0.0018 seconds
3_coll
- ./collisions_c 56 4 576 2 8 5
Total time for 5 iterations: 4.06642 seconds
Result (5.83115,-2.78179)
- slide 42
}

#### MPI
**Performance over multiple domains**
- message passing interface
- how to exchange information when there is no shared memory
- **Explicit data partitioning in MPI**
  - not easy like OpenMP
- MPI does not do auto syncronization

**MPI Mental Model**
1. How many processes
2. Which processors am I
3. Split the buffers
4. Iterate over my slice
5. Wait for the processes to finish

Example
- MPI_Recv & MPI_Send - Use conditional to operate on specific process and wait for signal
- MPI_Reduce - all processes must respond or main process hangs.

Can use both OpenMP and MPI together

**Running MPI Process groups**
- Single node MPI - mpiexec -n num-procs your-app
- SLURM-scheduled MPI - srun -n num-procs your-app

**Compilers and MPI**
- Does not understand natively
- complie w/

**MPI Boilerplate**
- Should run all the boilerplate functions in error checking blocks

**Collectives are your Friend**
- MPI_Bcast : brodcast - main process sends to all other processes
- MPI_Scatter: precompute values in master and send the right piece to the right one
- MPI_Gather: everyone computes separately and sends to the main process
- MPI_AllGather: 
- MPI_Allreduce:
- MPI_Alltoall: matrix transform/mult

**MPI-driven file access**
- One process distributes
- Every processes reads on it's own
- MPI-assisted IO collective functions
  - MPI_File_open,MPI_File_Read,MPI_File_write
    - Centralizes metadata and locking interactions
    - Aggregates IO operations on parallel filesystems
 
**Hands-On**
- 11_axpy_mpi
  - Please add MPI initialization and cleanup the following code

## Questions
- Support Model
  - Marty - Computational Physics - User Services
  - 1 FTE User Services and 1 FTE System Admin per system (Expanse, NRP)

- Datasets of interest
  - Alpha Fold
 
- How is Access Organized
  - 

- How many people support daily operations of Expanse
- What does support usually look like for an academic seeking help?
- Where does UCSD's support model for Expanse start and stop?
  - For people
  - For system maintenance
- What is the support volume like? (100-200 people on system a day)
- Do you have insight into how much code run on Expanse uses parallelism? Is the system being used efficiently?

- Do you have any monitoring in place that helps you understand how big of a problem NOT paralellizing is, on Expanse?

- How do you ensure

Maya - behind
Shante - helping cindy with admin
cindy and susan
linton
ethan? penn state astro infrastructure
planet formation (arizona - flag staff)
Vanti - 
Brian Campbell - humbolt, academic computing program, 2 roles, work with Mike on different classifications



# Take Back
- User Services 5 people / Sys Admins 5 people
- Access
  - Will direct to most appropriate HTP, HPC systems
  - Access Allocations are converted to system specific allocations 
- Setup Globus Endpoint - esNet data transfer nodes
- Create `expanse-client`
  - Is expanse-client - avaliable/open source?
- Request Trial Allocations at - consult@sdsc.edu
  - Explore - 1 paragraph - want to be on the system (400,000 credits)
  - Discover - 3 paragraphs
  - https://github.com/sdsc/sdsc-summer-institute-2026/blob/main/3.2_getting_help/HPC-DSI_Help_2026.pdf